# Vector stores and semantic search



In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Part I: Basic vector store implementation

In [2]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        if len(documents) == 0:
            return

        texts = [document.text for document in documents]
        new_embeddings = self.embedding_model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        self.documents.extend(documents)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if self.embeddings is None or len(self.documents) == 0:
            return []

        query_embedding = self.embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        scores = self.embeddings @ query_embedding
        top_indices = np.argsort(scores)[::-1][:top_k]

        results = []

        for index in top_indices:
            result = SearchResult(
                score=float(scores[index]),
                document=self.documents[index]
            )
            results.append(result)

        return results

In [3]:
dataset_path = Path("animal-fun-facts-dataset.csv")

if not dataset_path.exists():
    raise FileNotFoundError("No se encontró animal-fun-facts-dataset.csv en la misma carpeta del notebook.")

df_animals = pd.read_csv(dataset_path)
df_animals = df_animals.fillna("")

required_columns = ["animal_name", "source", "text", "media_link", "wikipedia_link"]
missing_columns = [column for column in required_columns if column not in df_animals.columns]

if len(missing_columns) > 0:
    raise ValueError(f"Faltan columnas en el dataset: {missing_columns}")

df_animals.head()

,animal_name,source,text,media_link,wikipedia_link
0,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"Aardvarks are sometimes called ""ant bears"", ""e...",,/wiki/Aardvark
1,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nhave rather primitive brains that a...,,/wiki/Aardvark
2,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Aardvarks\nteeth are lined with fine upright t...,,/wiki/Aardvark
3,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,"The aardvarks Latin family name ""Tubulidentata...",,/wiki/Aardvark
4,aardvark,https://www.animalfactsencyclopedia.com/Aardva...,Baby aardvarks are born with front teeth that ...,,/wiki/Aardvark


In [4]:
animal_documents = []

for _, row in df_animals.iterrows():
    text = str(row["text"])

    metadata = {
        "animal_name": str(row["animal_name"]),
        "source": str(row["source"]),
        "media_link": str(row["media_link"]),
        "wikipedia_link": str(row["wikipedia_link"])
    }

    document = Document(text=text, metadata=metadata)
    animal_documents.append(document)

print(f"Documentos cargados: {len(animal_documents)}")

Documentos cargados: 7734


In [5]:
vector_store = VectorStore(embedding_model)

vector_store.add_documents(animal_documents)

print(f"Documentos agregados al VectorStore: {len(vector_store.documents)}")
print(f"Forma de los embeddings: {vector_store.embeddings.shape}")

Documentos agregados al VectorStore: 7734
Forma de los embeddings: (7734, 384)


In [6]:
def mostrar_resultados(query: str, results: list[SearchResult]):
    print("=" * 100)
    print(f"Consulta: {query}")
    print("=" * 100)

    for i, result in enumerate(results, start=1):
        print(f"Resultado {i}")
        print(f"Score: {result.score:.4f}")
        print(f"Texto: {result.document.text}")
        print("Metadatos:")

        for key, value in result.document.metadata.items():
            print(f"  {key}: {value}")

In [7]:
queries = [
    "animals that can fly",
    "animals that live in the ocean",
    "dangerous predators",
    "animals with strong sense of smell",
    "animals that sleep a lot"
]

for query in queries:
    results = vector_store.search(query, top_k=5)
    mostrar_resultados(query, results)

Consulta: animals that can fly
Resultado 1
Score: 0.7076
Texto: They don’t fly, they glide.
The only mammal which can independently fly is the bat. Instead, colugas glide which works in the same way as a wingsuit.
Metadatos:
  animal_name: colugo (flying lemur)
  source: https://factanimal.com/colugo/
  media_link: 
  wikipedia_link: /wiki/Colugo
Resultado 2
Score: 0.6901
Texto: Not all birds are able to fly!
Metadatos:
  animal_name: bird
  source: https://a-z-animals.com/animals/bird/
  media_link: 
  wikipedia_link: /wiki/Bird
Resultado 3
Score: 0.6481
Texto: They rarely fly..
They move around on foot most of the time, only taking to the air to reach their nests or for courtship displays.
Metadatos:
  animal_name: secretary bird
  source: https://factanimal.com/secretarybird/
  media_link: 
  wikipedia_link: /wiki/Secretarybird
Resultado 4
Score: 0.6437
Texto: Bats are the only mammals with wings, and the only ones that can truly fly
Metadatos:
  animal_name: bat
  source: https://w

In [8]:
query_personalizada = "animals that are very intelligent"

results = vector_store.search(query_personalizada, top_k=5)
mostrar_resultados(query_personalizada, results)

Consulta: animals that are very intelligent
Resultado 1
Score: 0.7165
Texto: These dogs are very intelligent and are great with children.
Metadatos:
  animal_name: yoranian
  source: https://a-z-animals.com/animals/yoranian/
  media_link: 
  wikipedia_link: 
Resultado 2
Score: 0.6898
Texto: Meerkats are some of the most intelligent animals.
Meerkats are much more intelligent than they were initially given credit for. When you think of intelligent animals, you probably think about dogs, elephants, and dolphins.
Metadatos:
  animal_name: meerkat
  source: https://factanimal.com/meerkat/
  media_link: 
  wikipedia_link: /wiki/Meerkat
Resultado 3
Score: 0.6642
Texto: Highly active and intelligent dogs!
Metadatos:
  animal_name: bedlington terrier
  source: https://a-z-animals.com/animals/bedlington-terrier/
  media_link: 
  wikipedia_link: /wiki/Bedlington_Terrier
Resultado 4
Score: 0.6569
Texto: Elephants have the largest brain in all of the animal kingdom, and are smart too..
Their brain

## Part II: Filtering by metadata

In [9]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        if len(documents) == 0:
            return

        texts = [document.text for document in documents]

        new_embeddings = self.embedding_model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        self.documents.extend(documents)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(
        self,
        query: str,
        top_k: int = 5,
        metadata_filter: dict[str, str] | None = None
    ) -> list[SearchResult]:
        if self.embeddings is None or len(self.documents) == 0:
            return []

        if metadata_filter is None:
            candidate_indices = list(range(len(self.documents)))
        else:
            candidate_indices = []

            for index, document in enumerate(self.documents):
                matches_filter = True

                for key, value in metadata_filter.items():
                    if key not in document.metadata:
                        matches_filter = False
                        break

                    if str(document.metadata[key]) != str(value):
                        matches_filter = False
                        break

                if matches_filter:
                    candidate_indices.append(index)

        if len(candidate_indices) == 0:
            return []

        query_embedding = self.embedding_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        candidate_embeddings = self.embeddings[candidate_indices]
        scores = candidate_embeddings @ query_embedding

        sorted_positions = np.argsort(scores)[::-1][:top_k]

        results = []

        for position in sorted_positions:
            original_index = candidate_indices[position]

            result = SearchResult(
                score=float(scores[position]),
                document=self.documents[original_index]
            )

            results.append(result)

        return results

In [10]:
filtered_vector_store = FilteredVectorStore(embedding_model)

filtered_vector_store.add_documents(animal_documents)

print(f"Documentos agregados al FilteredVectorStore: {len(filtered_vector_store.documents)}")
print(f"Forma de los embeddings: {filtered_vector_store.embeddings.shape}")

Documentos agregados al FilteredVectorStore: 7734
Forma de los embeddings: (7734, 384)


In [11]:
results = filtered_vector_store.search(
    query="animals that are very intelligent",
    top_k=5,
    metadata_filter={"animal_name": "elephant"}
)

mostrar_resultados("animals that are very intelligent | filtro: elephant", results)

Consulta: animals that are very intelligent | filtro: elephant
Resultado 1
Score: 0.6569
Texto: Elephants have the largest brain in all of the animal kingdom, and are smart too..
Their brains can weigh up to a whopping 5.4kg. Size doesn’t necessarily equate directly to intelligence, however evidence suggests that elephants are some of the most intelligent, social and empathic animals on the planet.1
Metadatos:
  animal_name: elephant
  source: https://factanimal.com/elephants/
  media_link: 
  wikipedia_link: /wiki/Elephant
Resultado 2
Score: 0.4386
Texto: Elephants are the largest living land animals on earth..
The African bull elephant can grow as large as 13 feet (4 meters) tall, weigh between 4,000-7,500 kg and can have tusks as long as 6.5 feet (2 meters) in length weighing 100 pounds each (45 kg).
Metadatos:
  animal_name: elephant
  source: https://factanimal.com/elephants/
  media_link: 
  wikipedia_link: /wiki/Elephant
Resultado 3
Score: 0.4109
Texto: [Elephants and bees do no